Notebook to reshape population data results and run linear mixed models(supplement to population_analysis_summary.ipynb) with statsmodels


In [ ]:
import pandas as pd
import numpy as np
import os
import scipy.io
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import pickle

import statsmodels.formula.api as smf

In [ ]:
save_csvs=True

In [ ]:
decoding_performance_path = r"C:\miniscope_files\analyzed\xsessionxanimal_results\population_metrics\decoding\each_metric_across_sessions"

In [ ]:
#import the decoding performance summary files
#full neuron dataset

identity_full_res = pd.read_csv(os.path.join(decoding_performance_path,"identity_all_animals_all_sessions.csv"))
social_location_full_res = pd.read_csv(os.path.join(decoding_performance_path,"location_social_all_animals_all_sessions.csv"))
empty_location_full_res = pd.read_csv(os.path.join(decoding_performance_path,"location_empty1_all_animals_all_sessions.csv"))
social_location_full_res

In [ ]:
#within-animal subsampled dataset

identity_within_res = pd.read_csv(os.path.join(decoding_performance_path,"identity_within_all_animals_all_sessions.csv"))
social_location_within_res = pd.read_csv(os.path.join(decoding_performance_path,"location_social_within_all_animals_all_sessions.csv"))
empty_location_within_res = pd.read_csv(os.path.join(decoding_performance_path,"location_empty1_within_all_animals_all_sessions.csv"))

identity_within_res

In [ ]:
#filter out the sessions that did not meet pseudotrial criteria:

sesh_to_drop = pd.read_csv(os.path.join(r"C:\miniscope_files\analyzed\xsessionxanimal_results\population_metrics\decoding\pseudotrials_vs_decoding_performance","decoding_runs_below_training_data_threshold.csv"))
sesh_to_drop

In [ ]:
drop_set = set(zip(sesh_to_drop['animal'], sesh_to_drop['session_idx'], 
                   sesh_to_drop['decode_type'].str.replace('_across', '')))
drop_set

In [ ]:
identity_full_res_filt = identity_full_res[
    ~identity_full_res.apply(lambda r: (r['animal'], r['session_idx'], 'identity') in drop_set, axis=1)
].dropna(subset=['true_value'])

identity_within_res_filt = identity_within_res[
    ~identity_within_res.apply(lambda r: (r['animal'], r['session_idx'], 'identity') in drop_set, axis=1)
].dropna(subset=['true_value'])

identity_within_res_filt

In [ ]:
social_location_full_res_filt = social_location_full_res[
    ~social_location_full_res.apply(lambda r: (r['animal'], r['session_idx'], 'location_social') in drop_set, axis=1)
].dropna(subset=['true_value'])

social_location_within_res_filt = social_location_within_res[
    ~social_location_within_res.apply(lambda r: (r['animal'], r['session_idx'], 'location_social') in drop_set, axis=1)
].dropna(subset=['true_value'])

social_location_full_res_filt

In [ ]:
empty_location_full_res_filt = empty_location_full_res[
    ~empty_location_full_res.apply(lambda r: (r['animal'], r['session_idx'], 'location_empty1') in drop_set, axis=1)
].dropna(subset=['true_value'])

empty_location_within_res_filt = empty_location_within_res[
    ~empty_location_within_res.apply(lambda r: (r['animal'], r['session_idx'], 'location_empty1') in drop_set, axis=1)
].dropna(subset=['true_value'])

empty_location_within_res_filt

In [ ]:
empty_location_full_res_filt

In [ ]:
def reshape_true_vs_null(df, decoder_label, data_type='full'):
    """
    Take a decoding results DataFrame and reshape into long format with one 'true' row and one 'null' row per (animal, session).
    """
    true_rows = df[['animal', 'session_idx', 'short_label', 'true_value']].copy()
    true_rows['source'] = 'true'
    true_rows['accuracy'] = true_rows['true_value']

    if data_type=='full':
        null_rows = df[['animal', 'session_idx', 'short_label', 'null_mean']].copy()
        null_rows['source'] = 'null'
        null_rows['accuracy'] = null_rows['null_mean']
    
        long = pd.concat([
            true_rows[['animal', 'session_idx', 'short_label', 'source', 'accuracy']],
            null_rows[['animal', 'session_idx', 'short_label', 'source', 'accuracy']]
        ], ignore_index=True)
    else:
        long = true_rows[['animal', 'session_idx', 'short_label', 'source', 'accuracy']]
    long['decoder'] = decoder_label
    return long.dropna(subset=['accuracy'])

In [ ]:
identity_full_filt_true_vs_null_df = reshape_true_vs_null(identity_full_res_filt,decoder_label='identity_full_filtered',data_type='full')
social_location_full_filt_true_vs_null_df = reshape_true_vs_null(social_location_full_res_filt,decoder_label='social_location_full_filtered',data_type='full')
empty_location_full_res_filt_true_vs_null_df = reshape_true_vs_null(empty_location_full_res_filt,decoder_label='empty_location_full_filtered',data_type='full')

In [ ]:
social_location_full_filt_true_vs_null_df

In [ ]:
def true_vs_null_LMM(df):
    df1=df.copy()

    df1['source']=pd.Categorical(df1['source'],categories=['null','true'])
    assert df1['source'].isna().sum() == 0, \
        "NaN in source after categorical conversion - check source values are 'true' or 'null'"
    
    df1['session_idx'] = pd.Categorical(df1['session_idx'].astype(int).astype(str), 
                                    categories=['1','2','3','4','5'])
    assert df1['session_idx'].isna().sum() == 0, "NaN in session_idx after categorical conversion - check session indices are 1-5"

    m = smf.mixedlm(
        "accuracy ~ source * session_idx",
        data=df1,
        groups=df1["animal"]
    )

    result = m.fit()
    print(result.summary())

    #joint F-tests for each effect
    interaction_names = [n for n in result.fe_params.index if ':' in n]
    session_main_names = [n for n in result.fe_params.index 
                       if 'session_idx' in n and ':' not in n]
    source_main_names = [n for n in result.fe_params.index 
                       if 'source' in n and ':' not in n]

    #f tests test the null hypothesis that the corresponding coefficients/regressors are all = 0 (i.e., no effect on the dependent variable)
    f_test_source = result.f_test(source_main_names)
    f_test_sesh = result.f_test(session_main_names)
    f_test_int = result.f_test(interaction_names)

    print("\n=== Joint F-tests ===")
    print("Source main effect: p=", f_test_source.pvalue)
    print("Session main effect: p=", f_test_sesh.pvalue)
    print("Interaction: p=", f_test_int.pvalue)


    results_dict = {
        'source_main_F': f_test_source.fvalue,
        'source_main_p': f_test_source.pvalue,
        'source_main_df_num': f_test_source.df_num,
        'source_main_df_denom': f_test_source.df_denom,
        'session_main_F': f_test_sesh.fvalue,
        'session_main_p': f_test_sesh.pvalue,
        'session_main_df_num': f_test_sesh.df_num,
        'session_main_df_denom': f_test_sesh.df_denom,
        'interaction_F': f_test_int.fvalue,
        'interaction_p': f_test_int.pvalue,
        'interaction_df_num': f_test_int.df_num,
        'interaction_df_denom': f_test_int.df_denom,
        'n_obs': result.nobs,
        'random_effect_var_animal': result.cov_re.iloc[0, 0],
        'residual_var': result.scale,
    }
    
    # Get the non-reference source name (the one statsmodels reports the contrast for)
    source_coef_name = [n for n in result.fe_params.index 
                     if 'source' in n and ':' not in n and '[T.' in n][0]

    results_dict.update({
        'source_coef_name': source_coef_name,
        'source_coef': result.fe_params[source_coef_name],
        'source_coef_se': result.bse_fe[source_coef_name],
        'source_coef_ci_low': result.conf_int().loc[source_coef_name, 0],
        'source_coef_ci_high': result.conf_int().loc[source_coef_name, 1],
    })
    
    return results_dict

In [ ]:
true_vs_null_results = []

In [ ]:
identity_true_vs_null_result = true_vs_null_LMM(identity_full_filt_true_vs_null_df)
identity_true_vs_null_result['decoder_label'] = 'identity'
true_vs_null_results.append(identity_true_vs_null_result)

In [ ]:
social_location_true_vs_null_result = true_vs_null_LMM(social_location_full_filt_true_vs_null_df)
social_location_true_vs_null_result['decoder_label'] = 'social_location'
true_vs_null_results.append(social_location_true_vs_null_result)

In [ ]:
empty_location_true_vs_null_result = true_vs_null_LMM(empty_location_full_res_filt_true_vs_null_df)
empty_location_true_vs_null_result['decoder_label'] = 'empty_location'
true_vs_null_results.append(empty_location_true_vs_null_result)

In [ ]:
true_vs_null_results_df = pd.DataFrame(true_vs_null_results)

if save_csvs:
    true_vs_null_results_df.to_csv(os.path.join(decoding_performance_path,'all_true_vs_null_lmm_results.csv'), index=False)

true_vs_null_results_df


In [ ]:
#location vs. social decoding and empty location vs social location

In [ ]:
#because there are no nulls for the _within dfs, this will just give us the true values
identity_within_filt_true_df = reshape_true_vs_null(identity_within_res_filt,decoder_label='identity_within_filtered',data_type='within')
social_location_within_filt_true_df = reshape_true_vs_null(social_location_within_res_filt,decoder_label='social_location_within_filtered',data_type='within')
empty_location_within_filt_true_df = reshape_true_vs_null(empty_location_within_res_filt,decoder_label='empty_location_within_filtered',data_type='within')
#identity_within_filt_true_only = identity_within_filt_true_vs_null_df[identity_within_filt_true_vs_null_df['source']=='true']
social_location_within_filt_true_df

In [ ]:
#combine these for LMMs comparing location vs. identity

identity_vs_social_location_df = pd.concat([identity_within_filt_true_df,social_location_within_filt_true_df]).reset_index(drop=True)
identity_vs_social_location_df

In [ ]:
empty_vs_social_location_df = pd.concat([empty_location_within_filt_true_df,social_location_within_filt_true_df]).reset_index(drop=True)
empty_vs_social_location_df

In [ ]:
def two_decode_types_LMM(df,categories_list):
    df1=df.copy()
    df1['decoder']=pd.Categorical(df1['decoder'],categories=categories_list)
    assert df1['decoder'].isna().sum() == 0, "NaN in decoder after categorical conversion - check categories_list matches input data"
    
    df1['session_idx'] = pd.Categorical(df1['session_idx'].astype(int).astype(str), 
                                       categories=['1','2','3','4','5'])
    assert df1['session_idx'].isna().sum() == 0, "NaN in session_idx after categorical conversion - check session indices are 1-5"
    

    m = smf.mixedlm(
        "accuracy ~ decoder * session_idx",
        data=df1,
        groups=df1["animal"]
    )

    result = m.fit()
    print(result.summary())

    #joint F-tests for each effect
    interaction_names = [n for n in result.fe_params.index if ':' in n]
    session_main_names = [n for n in result.fe_params.index 
                       if 'session_idx' in n and ':' not in n]
    decoder_main_names = [n for n in result.fe_params.index 
                       if 'decoder' in n and ':' not in n]

    f_test_d = result.f_test(decoder_main_names)
    f_test_s = result.f_test(session_main_names)
    f_test_i = result.f_test(interaction_names)

    print("\n=== Joint F-tests ===")
    print("Decoder main effect: p=", f_test_d.pvalue)
    print("Session main effect: p=", f_test_s.pvalue)
    print("Interaction: p=", f_test_i.pvalue)


    results_dict = {
        'decoder_main_F': f_test_d.fvalue,
        'decoder_main_p': f_test_d.pvalue,
        'decoder_main_df_num': f_test_d.df_num,
        'decoder_main_df_denom': f_test_d.df_denom,
        'session_main_F': f_test_s.fvalue,
        'session_main_p': f_test_s.pvalue,
        'session_main_df_num': f_test_s.df_num,
        'session_main_df_denom': f_test_s.df_denom,
        'interaction_F': f_test_i.fvalue,
        'interaction_p': f_test_i.pvalue,
        'interaction_df_num': f_test_i.df_num,
        'interaction_df_denom': f_test_i.df_denom,
        'n_obs': result.nobs,
        'random_effect_var_animal': result.cov_re.iloc[0, 0],
        'residual_var': result.scale,
    }
    
    # Get the non-reference decoder name (the one statsmodels reports the contrast for)
    decoder_coef_name = [n for n in result.fe_params.index 
                     if 'decoder' in n and ':' not in n and '[T.' in n][0]

    results_dict.update({
        'decoder_coef_name': decoder_coef_name,
        'decoder_coef': result.fe_params[decoder_coef_name],
        'decoder_coef_se': result.bse_fe[decoder_coef_name],
        'decoder_coef_ci_low': result.conf_int().loc[decoder_coef_name, 0],
        'decoder_coef_ci_high': result.conf_int().loc[decoder_coef_name, 1],
    })
    
    return results_dict

In [ ]:
two_decode_types_results = []

In [ ]:
id_vs_social_location_res = two_decode_types_LMM(identity_vs_social_location_df,['identity_within_filtered','social_location_within_filtered'])
id_vs_social_location_res['decoder_comparison'] = 'identity_vs_social_location'
two_decode_types_results.append(id_vs_social_location_res)

In [ ]:
empty_vs_social_location_res = two_decode_types_LMM(empty_vs_social_location_df,['empty_location_within_filtered','social_location_within_filtered'])
empty_vs_social_location_res['decoder_comparison'] = 'empty_vs_social_location'
two_decode_types_results.append(empty_vs_social_location_res)

In [ ]:
two_decode_types_results_df = pd.DataFrame(two_decode_types_results)

if save_csvs:
   two_decode_types_results_df.to_csv(os.path.join(decoding_performance_path,'decoder_type_variable_LMM_results.csv'), index=False)

two_decode_types_results_df


In [ ]:
#import mahalanobis distance and run identity vs locatoin and socila location vs empty location

In [ ]:
MD_path = r"C:\miniscope_files\analyzed\xsessionxanimal_results\population_metrics\mahalanobis_distance"

In [ ]:
#import the decoding performance summary files
#full neuron dataset

#being lazy and going to use the same LMM function for variable comparisons as for decoding -- which means we need to change a couple value names

identity_MD_res = pd.read_csv(os.path.join(MD_path,"md_ratio_identity_all_sessions.csv")).dropna(subset='value').rename(columns={'value': 'accuracy'})
social_location_MD_res = pd.read_csv(os.path.join(MD_path,"md_ratio_social_location_all_sessions.csv")).dropna(subset='value').rename(columns={'value': 'accuracy'})
empty_location_MD_res = pd.read_csv(os.path.join(MD_path,"md_ratio_empty_location_all_sessions.csv")).dropna(subset='value').rename(columns={'value': 'accuracy'})
empty_location_MD_res

In [ ]:
identity_MD_res['decoder']='identity'
social_location_MD_res['decoder']='social_location'
empty_location_MD_res['decoder']='empty_location'

identity_MD_res

In [ ]:
id_vs_soc_loc_md = pd.concat([identity_MD_res, social_location_MD_res], ignore_index=True)
empty_vs_soc_loc_md = pd.concat([empty_location_MD_res, social_location_MD_res], ignore_index=True)
id_vs_soc_loc_md

In [ ]:
MD_comps_results = []

In [ ]:
id_vs_soc_loc_MD_res = two_decode_types_LMM(id_vs_soc_loc_md,['identity','social_location'])
id_vs_soc_loc_MD_res['MD_type_comparison'] = 'identity_vs_social_location'
MD_comps_results.append(id_vs_soc_loc_MD_res)

In [ ]:
empty_vs_soc_loc_MD_res = two_decode_types_LMM(empty_vs_soc_loc_md,['empty_location','social_location'])
empty_vs_soc_loc_MD_res['MD_type_comparison'] = 'empty_vs_social_location'
MD_comps_results.append(empty_vs_soc_loc_MD_res)

In [ ]:
two_MD_types_results_df = pd.DataFrame(MD_comps_results)

if save_csvs:
   two_MD_types_results_df.to_csv(os.path.join(MD_path,'MD_ratio_type_comparison_LMM_results.csv'), index=False)

two_MD_types_results_df
